# init data

## datasets

In [4]:
import pandas as pd

strongs_df = pd.read_csv("strongs.csv")
lv65_df = pd.read_csv("latvian_full65.csv")
import unicodedata
def raccs_common(text):
    d = {
        #ord('\N{COMBINING ACUTE ACCENT}'):None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFC', text).translate(d)
print(strongs_df["form"].apply(raccs_common).head())

0       Παῦλος
1       κλητὸς
2    ἀπόστολος
3      Χριστοῦ
4        Ἰησοῦ
Name: form, dtype: str


## prep grouped

In [8]:
def prepare_grouped(strongs_df, lv_df):
    strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
    lv_g = lv_df.groupby(["book", "chapter", "verse"])
    return strongs_g, lv_g

strongs_g, lv_g = prepare_grouped(strongs_df, lv65_df)

## select data

In [40]:
import os
import re
import json
import time
from pathlib import Path
from collections import Counter
from bs4 import BeautifulSoup

def validate_latvian_usage(verse_text, mappings, leftover_list):
    """
    Checks if all words in the reference Latvian text are present 
    either in the Greek-word mappings or the leftover list.
    Case-sensitive as per user request.
    """
    # Tokenize reference text (preserving case)
    text_words = re.findall(r'\b\w+\b', verse_text)
    
    # Collect all words used in mappings
    mapped_words = []
    for m in mappings:
        if m.get('latvian'):
            # If latvian mapping is a list, extend; if string, append
            #if isinstance(m['latvian'], list):
            for val in m['latvian']:
                mapped_words.extend(re.findall(r'\b\w+\b', val))
#            else:
#                mapped_words.extend(re.findall(r'\b\w+\b', m['latvian']))
    
    # Also add words already in the leftover list
    for entry in leftover_list:
        mapped_words.extend(re.findall(r'\b\w+\b', entry))
    
    text_counts = Counter(text_words)
    mapped_counts = Counter(mapped_words)

    missing = []
    for word, count in text_counts.items():
        if mapped_counts[word] < count:
            # Add each missing instance to the list
            for _ in range(count - mapped_counts[word]):
                missing.append(word)
            
    return missing

def split_chapter_html(book, chapter_num, strongs_g, lv_g):
    source_path = Path("bible") / book.lower() / f"{chapter_num}.html"
    
    if not source_path.exists():
        print(f"Source {source_path} not found.")
        return

    with open(source_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Create destination: bible/matthew/15/
    dest_dir = Path("bible") / book.lower() / str(chapter_num)
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Each verse is in a <div class="verse-container">
    verse_containers = soup.find_all("div", class_="verse-container")

    #check numbered case
    numcase = False
    # first mapping-table div
    mapping_div = soup.find("table", class_="mapping-table")
    if mapping_div:
        first_th = mapping_div.find("th")
        if first_th:
            numcase = first_th.get_text(strip=True) =="#"
    #print(numcase)
            
    for container in verse_containers:
        # Extract verse number from the title "Verse 1"
        title_cont_style_1 = container.find("div", class_="verse-title")
        title_cont_style_2 = container.find("div", class_="verse-header")
        title_text = (title_cont_style_1 if title_cont_style_1 else title_cont_style_2).get_text()
        v_match = re.search(r'Verse\s+(\d+)', title_text) if title_cont_style_1 \
        else re.search(r'\s+\d:(\d+)', title_text)
        if not v_match: continue
        v_num = int(v_match.group(1))
        
        #key = (book.capitalize(), int(chapter_num), v_num)
        key = (book, int(chapter_num), v_num)

        # 1. Reference Data Lookup
        if key not in strongs_g.groups or key not in lv_g.groups:
            print(f"⚠️ No reference for {key}. Skipping.")
            continue

        ref_strongs = strongs_g.get_group(key).sort_values("word")
        ref_latvian_text = lv_g.get_group(key).iloc[0]["text"]

        # 2. Extract Mappings from Table
        mapping_greek_words = []
        rows = container.find("tbody").find_all("tr")
        
        for idx, row in enumerate(rows):
            cols = row.find_all("td")
            #if len(cols) < 3: continue
            if len(cols) < 3 if numcase else 2: continue
            
            greek_word = (cols[1] if numcase else cols[0]).get_text().strip()
            #print(cols[1].get_text())
            #print(cols[2].get_text())
            lget = (cols[2] if numcase else cols[1]).get_text().strip()
            latvian_val = (lget if lget != "-" else "").split()
#            strong_num = cols[2].get_text().strip()
            
            # Formatting as per your expected JSON structure
            mapping_greek_words.append({
                "index": idx,
                "greek": greek_word,
                "latvian": latvian_val,
 #               "strong": strong_num
            })

        # 3. Check for Notes/Implied words (Leftovers)
        leftover_latvian = []
        note_div = container.find("div", class_="note")
        if note_div:
            # You can extract specific implied words if your notes follow a pattern
            # For now, we rely on the validation to find what's missing
            pass

        # 4. Validation & Auto-fill Leftovers
        if len(ref_strongs) != len(mapping_greek_words):
            print(f"❌ Greek count mismatch at {key}: {len(ref_strongs)} vs {len(mapping_greek_words)}")

        missing_from_text = validate_latvian_usage(ref_latvian_text, mapping_greek_words, leftover_latvian)
        leftover_latvian.extend(missing_from_text)

        # 5. Save Output
        verse_output = {
            "locus": f"{book.lower()}_{chapter_num}_{v_num}",
            "greek_words": mapping_greek_words,
            "leftover_latvian": leftover_latvian
        }

        file_name = f"{book.lower()}_{chapter_num}_{v_num}.txt"
        with open(dest_dir / file_name, "w", encoding="utf-8") as out_f:
            json.dump(verse_output, out_f, ensure_ascii=False, indent=4)

    print(f"✅ Processed {len(verse_containers)} verses from {source_path}")

# Example Usage:
# split_chapter_html("matthew", 15, strongs_g, lv_g)


# doit()

In [42]:
for i in range (1, 2):#29):
    split_chapter_html("matthew", i, strongs_g, lv_g)

✅ Processed 25 verses from bible/matthew/1.html
❌ Greek count mismatch at ('matthew', 2, 2): 20 vs 21
❌ Greek count mismatch at ('matthew', 2, 3): 11 vs 13
❌ Greek count mismatch at ('matthew', 2, 4): 16 vs 17
❌ Greek count mismatch at ('matthew', 2, 7): 14 vs 15
❌ Greek count mismatch at ('matthew', 2, 8): 22 vs 24
❌ Greek count mismatch at ('matthew', 2, 9): 25 vs 26
❌ Greek count mismatch at ('matthew', 2, 10): 8 vs 10
❌ Greek count mismatch at ('matthew', 2, 11): 30 vs 31
❌ Greek count mismatch at ('matthew', 2, 12): 16 vs 17
❌ Greek count mismatch at ('matthew', 2, 13): 40 vs 41
❌ Greek count mismatch at ('matthew', 2, 15): 23 vs 24
❌ Greek count mismatch at ('matthew', 2, 16): 37 vs 38
❌ Greek count mismatch at ('matthew', 2, 18): 20 vs 21
❌ Greek count mismatch at ('matthew', 2, 19): 14 vs 15
❌ Greek count mismatch at ('matthew', 2, 20): 22 vs 23
❌ Greek count mismatch at ('matthew', 2, 22): 25 vs 27
❌ Greek count mismatch at ('matthew', 2, 23): 17 vs 18
✅ Processed 23 verses fr

KeyboardInterrupt: 